# Polymarket Event Impact Trading - Research Notebook

This notebook demonstrates the complete workflow for training and backtesting the event-driven trading strategy.

## Strategy Overview

**Strategy #2: Deep Learning Event Impact Forecasting**

- Detects breaking news and events using multiple data sources
- Extracts features from events (sentiment, credibility, timing)
- Combines with market features (price, volume, order book)
- Predicts price movement direction after events
- Executes trades with confidence-based position sizing

## Workflow

1. **Data Collection**: Gather historical events and price data
2. **Feature Engineering**: Extract predictive features
3. **Model Training**: Train ML models to predict price movements
4. **Backtesting**: Validate strategy on historical data
5. **Live Trading**: Deploy bot with risk management

In [3]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Local imports
from polymarket_client import PolymarketClient, MarketFilter
from event_detector import EventDetector, Event
from feature_extractor import FeatureEngineering, SentimentAnalyzer
from models import PriceMovementPredictor, EnsemblePredictor, TradingSignalGenerator
from backtester import Backtester, create_sample_signals, create_sample_prices

print("✓ Imports successful")

✓ Imports successful


## 1. Initialize Clients

In [4]:
# Initialize Polymarket client (no API key needed for public data)
client = PolymarketClient()

# Get active markets
markets = client.get_markets(limit=50, active=True)
print(f"Found {len(markets)} active markets")

# Display sample market
if markets:
    sample_market = markets[0]
    print(f"\nSample Market:")
    print(f"Question: {sample_market.get('question', 'N/A')}")
    print(f"Volume: ${sample_market.get('volume', 0):,.2f}")
    print(f"End Date: {sample_market.get('end_date_iso', 'N/A')}")

Found 50 active markets

Sample Market:
Question: US recession in 2025?


ValueError: Unknown format code 'f' for object of type 'str'

## 2. Event Detection Demo

In [ ]:
# Initialize event detector with RSS feeds (no API key required)
rss_feeds = [
    "https://www.coindesk.com/arc/outboundfeeds/rss/",
    "https://www.reuters.com/rssFeed/topNews"
]

event_detector = EventDetector(rss_feeds=rss_feeds)

# Get recent events
events = event_detector.get_all_recent_events(lookback_hours=24)
print(f"Found {len(events)} recent events\n")

# Display recent events
for i, event in enumerate(events[:5]):
    print(f"{i+1}. {event.title[:80]}...")
    print(f"   Source: {event.source}, Time: {event.published_time}")
    print(f"   Keywords: {', '.join(event.keywords[:5])}\n")

## 3. Feature Extraction Demo

In [ ]:
# Create sample event
sample_event = Event(
    title="Bitcoin surges to new all-time high amid institutional buying",
    description="Bitcoin price reached $75,000 as major institutions announced new cryptocurrency allocations.",
    source="Bloomberg",
    published_time=datetime.now(),
    url="https://example.com",
    keywords=["Bitcoin", "cryptocurrency", "institutional", "surge", "price"]
)

# Extract features
feature_eng = FeatureEngineering()
event_features = feature_eng.event_extractor.extract_event_features(sample_event)

print("Event Features:")
for key, value in event_features.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

## 4. Generate Synthetic Training Data

For demonstration purposes, we'll create synthetic data. In production, you would:
1. Collect historical events from news archives
2. Match them to Polymarket markets
3. Label with actual price movements

In [ ]:
# Generate synthetic training data
np.random.seed(42)

n_samples = 1000
feature_names = [
    'event_sentiment_score', 'event_sentiment_magnitude', 'event_source_credibility',
    'market_current_price', 'market_price_volatility', 'market_price_trend',
    'market_total_volume', 'orderbook_spread', 'orderbook_depth_imbalance'
]

# Create features
X_train = pd.DataFrame(
    np.random.randn(n_samples, len(feature_names)),
    columns=feature_names
)

# Normalize some features to realistic ranges
X_train['event_sentiment_score'] = np.random.uniform(-1, 1, n_samples)
X_train['event_source_credibility'] = np.random.uniform(0.3, 1.0, n_samples)
X_train['market_current_price'] = np.random.uniform(0.2, 0.8, n_samples)
X_train['orderbook_spread'] = np.random.uniform(0.01, 0.1, n_samples)

# Create labels (1: up, 0: no change, -1: down)
# Positive sentiment -> more likely to go up
probs = X_train['event_sentiment_score'].apply(
    lambda x: [0.2, 0.3, 0.5] if x > 0.3 else [0.5, 0.3, 0.2] if x < -0.3 else [0.33, 0.34, 0.33]
)
y_train = np.array([np.random.choice([1, 0, -1], p=p) for p in probs])

print(f"Training data shape: {X_train.shape}")
print(f"Label distribution: {pd.Series(y_train).value_counts().to_dict()}")

## 5. Train Models

In [ ]:
# Train Random Forest model
print("Training Random Forest...")
rf_model = PriceMovementPredictor(model_type='random_forest')
rf_metrics = rf_model.train(X_train, y_train, validation_split=0.2)

print(f"\nRandom Forest Performance:")
print(f"  Train Accuracy: {rf_metrics['train_accuracy']:.4f}")
print(f"  Val Accuracy: {rf_metrics['val_accuracy']:.4f}")
print(f"  Val F1: {rf_metrics['val_f1']:.4f}")

In [ ]:
# Feature importance
feature_importance = rf_model.get_feature_importance()
print("\nTop 5 Most Important Features:")
print(feature_importance.head())

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'][:10], feature_importance['importance'][:10])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Train Ensemble model
print("Training Ensemble Model...")
ensemble = EnsemblePredictor(
    model_types=['random_forest', 'gradient_boost', 'logistic']
)
ensemble_metrics = ensemble.train(X_train, y_train, validation_split=0.2)

print(f"\nEnsemble Performance:")
print(f"  Ensemble Accuracy: {ensemble_metrics['ensemble_accuracy']:.4f}")
print(f"  Model Weights: {[f'{w:.3f}' for w in ensemble_metrics['weights']]}")

## 6. Save Trained Model

In [ ]:
# Save the best model
model_path = 'trained_model.pkl'
rf_model.save(model_path)
print(f"Model saved to {model_path}")

## 7. Backtesting

In [ ]:
# Create sample signals for backtesting
signals_df = create_sample_signals(n_signals=200, n_markets=20)
print(f"Generated {len(signals_df)} signals")

# Create sample price data
market_ids = signals_df['market_id'].unique()
start_time = signals_df['timestamp'].min()
end_time = signals_df['timestamp'].max()

price_data = create_sample_prices(market_ids, start_time, end_time)
print(f"Generated price data for {len(price_data)} markets")

In [ ]:
# Run backtest
backtester = Backtester(
    initial_capital=10000,
    position_size=100,
    hold_time_hours=24,
    max_positions=10
)

results = backtester.run_backtest(signals_df, price_data)

print("\n" + "="*50)
print("BACKTEST RESULTS")
print("="*50)
for key, value in results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Plot backtest results
backtester.plot_results()

## 8. Signal Generation Demo

In [ ]:
# Create signal generator
signal_gen = TradingSignalGenerator(
    model=rf_model,
    min_confidence=0.65,
    min_expected_return=0.03
)

# Generate signal for sample features
sample_features = X_train.iloc[[0]].copy()
current_price = 0.45

signal = signal_gen.generate_signal(sample_features, current_price)

print("Trading Signal:")
for key, value in signal.items():
    print(f"  {key}: {value}")

## 9. Performance Analysis

In [ ]:
# Analyze backtest trades
if backtester.portfolio.closed_trades:
    trades_df = pd.DataFrame([t.to_dict() for t in backtester.portfolio.closed_trades])
    
    # Distribution of returns
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Return distribution
    axes[0].hist(trades_df['return_pct'], bins=30, edgecolor='black')
    axes[0].axvline(x=0, color='r', linestyle='--', label='Break-even')
    axes[0].set_xlabel('Return (%)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Trade Returns')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # PnL by side
    trades_df.groupby('side')['pnl'].sum().plot(kind='bar', ax=axes[1])
    axes[1].set_xlabel('Trade Side')
    axes[1].set_ylabel('Total PnL ($)')
    axes[1].set_title('Total PnL by Trade Side')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\nTrade Statistics:")
    print(trades_df[['pnl', 'return_pct', 'duration_hours']].describe())

## 10. Next Steps

### For Real Trading:

1. **Get API Keys**:
   - Polymarket API access
   - NewsAPI key (https://newsapi.org)
   - Optional: Twitter API access

2. **Collect Real Data**:
   - Historical event data from news archives
   - Historical Polymarket price data
   - Match events to markets and label outcomes

3. **Retrain Models**:
   - Train on real data instead of synthetic
   - Optimize hyperparameters
   - Cross-validate on different time periods

4. **Paper Trading**:
   - Run `trader.py` with `paper_trading=True`
   - Monitor performance for 1-2 weeks
   - Adjust risk parameters

5. **Live Trading**:
   - Start with small position sizes
   - Monitor closely
   - Scale up gradually

### Improvements to Consider:

- **Better Sentiment**: Use FinBERT or GPT-4 for sentiment
- **More Features**: Add social media sentiment, poll data
- **Multi-Agent**: Implement Strategy #1 (ensemble of specialized agents)
- **Market Making**: Add Strategy #3 (RL market maker)
- **Risk Management**: Implement correlation-based position limits
- **Execution**: Optimize order placement and slippage

In [ ]:
print("\n" + "="*60)
print("Research notebook complete!")
print("="*60)
print("\nNext: Edit config.json and run trader.py for live trading")